ESQUELETO:

Este es el archivo para subir el modelo a la nube.


CÓMO FUNCIONA:

1. Tomar un modelo ya realizado.

2. Lo sube a la nube.

ASEGÚRENSE DE QUE SE COORDINAN LO NECESARIO PARA QUE LOS NOMBRES COINCIDAN (no es de chill cambiar el estándar xd).

In [1]:
## STEP 0: Install/verify required packages in this notebook kernel (run this first).
import sys
import subprocess

required_packages = [
    "azureml-core",
    "azureml-defaults",
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", *required_packages])
print("Dependencies installed for notebook execution.")

Dependencies installed for notebook execution.


In [2]:
## STEP 1: Set up the workspace.

import json

#GETTING MY ID:
id = open('credentials.json', 'r')
mi = json.load(id)
my_id = mi["subscription_id"],

from azureml.core import Workspace

ws = Workspace.create(name="workspace4",
                      subscription_id=my_id[0],  # my_id is a tuple; AzureML expects a string
                      resource_group="Papus",
                      location="eastus")

from azureml.core.model import Model

mname = "model"
registered_model = Model.register(model_path="model.pkl",
                                  model_name=mname,
                                  workspace=ws)

Deploying StorageAccount with name workspacstoragec5754ccbe.
Deploying KeyVault with name workspackeyvault74169155.
Deploying AppInsights with name workspacinsights8669f52f.
Deployed AppInsights with name workspacinsights8669f52f. Took 1.41 seconds.
Deployed StorageAccount with name workspacstoragec5754ccbe. Took 20.65 seconds.
Deployed KeyVault with name workspackeyvault74169155. Took 17.54 seconds.
Deploying Workspace with name workspace4.
Deployed Workspace with name workspace4. Took 16.57 seconds.
Registering model model


In [6]:
################ score.py ###################

umb = open("umbral.json", "r")
umb = json.load(umb)
umbral = umb["umbral"][0]

scorepy = f"""
import json
import joblib
import numpy as np
import pandas as pd
from azureml.core.model import Model

def init():
  global model
  model_path = Model.get_model_path('{mname}')
  model = joblib.load(model_path)

def sigmoid(x):
  return [1 / (1 + np.exp(-y)) for y in x]

def run(raw_data):
  try: ## Try la predicción.
    data = json.loads(raw_data)['data'][0]
    data = pd.DataFrame(data)

    data_dummies = data.drop(["RowNumber", "CustomerId", "Surname", "Geography", "Gender", "Card Type"], axis=1) ## INCLUIR AQUÍ LO DE ZORROUNO

    result = model.predict(data_dummies).tolist()
    result_sigmoid = sigmoid(result)
    umbral = {umbral}
    result_finals = [1 if x > umbral else 0 for x in result_sigmoid]

    return json.dumps(result_finals)
  except Exception as e:
    return json.dumps(str(e))
"""

file_score = open("score.py", "w")
file_score.write(scorepy)
file_score.close()

In [ ]:
## STEP 2: Inference configuration: the blueprints for the German carpenter about how to build your furniture.

from azureml.core.environment import Environment
virtual_env = Environment("env-4-housing")

from azureml.core.conda_dependencies import CondaDependencies
virtual_env.python.conda_dependencies = CondaDependencies.create(
    conda_packages=['pandas', 'numpy', 'scikit-learn'],
    pip_packages=['joblib', 'azureml-defaults']
)  ## Asegúrense de coordinarse con el departamento de modelos para incluir los paquetes correctos. Si no, BOOM!


from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice
inference_config = InferenceConfig(
                                environment=virtual_env,
                                entry_script="score.py",
                                )
aci_config = AciWebservice.deploy_configuration(cpu_cores=0.5, memory_gb=1) ## ASEGÚRENSE DE ASIGNAR SUFICIENTE MADERA PARA SUS MUEBLES.

service = Model.deploy(workspace=ws,
                       name='service',
                       models=[registered_model],
                       inference_config=inference_config,
                       deployment_config=aci_config,
                       overwrite=True,
                       )

C:\Users\asgar\AppData\Local\Temp\ipykernel_18808\944137596.py:18: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(workspace=ws,


In [ ]:
## STEP 2.5: Retrieve assets and ensure deployment environment has required packages
import json
from azureml.core import Workspace, Model, Environment
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice, Webservice
from azureml.core.conda_dependencies import CondaDependencies

with open("credentials.json", "r") as f:
    creds = json.load(f)

subscription_id = creds["subscription_id"]
resource_group = "Papus"
workspace_name = "workspace3"
model_name = "model"
base_environment_name = "AzureML-ACPT-pytorch-1.13-py38-cuda11.7-gpu"
environment_name = "azureml-acpt-pytorch-1.13-py38-cuda11.7-gpu-inference"
service_name = "service"

# Connect to existing workspace
ws = Workspace.get(
    name=workspace_name,
    subscription_id=subscription_id,
    resource_group=resource_group,
    )

# Retrieve latest registered model version by name
registered_model = Model(ws, name=model_name)

# Reuse custom environment if it already exists
env_map = Environment.list(workspace=ws)
if environment_name in env_map:
    virtual_env = Environment.get(workspace=ws, name=environment_name)
    print(f"Using existing environment: {virtual_env.name}:{virtual_env.version}")
else:
    # Build a reusable custom environment once from the base curated image
    base_env = Environment.get(workspace=ws, name=base_environment_name)
    virtual_env = base_env.clone(environment_name)

    deps = virtual_env.python.conda_dependencies
    if deps is None:
        deps = CondaDependencies()

    for pkg in ["joblib", "pandas", "numpy", "scikit-learn", "azureml-defaults"]:
        deps.add_pip_package(pkg)

    virtual_env.python.conda_dependencies = deps
    virtual_env.register(workspace=ws)
    print(f"Registered new environment: {virtual_env.name}:{virtual_env.version}")

inference_config = InferenceConfig(
    environment=virtual_env,
    entry_script="score.py",
    )
aci_config = AciWebservice.deploy_configuration(cpu_cores=1, memory_gb=2)

# Optional: cleanly replace existing endpoint if it already exists
try:
    existing_service = Webservice(workspace=ws, name=service_name)
    existing_service.delete()
except Exception:
    pass

service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[registered_model],
    inference_config=inference_config,
    deployment_config=aci_config,
    overwrite=True,
    )

Exception: Error registering the environment definition. Code: 400
: {
  "error": {
    "code": "ValidationError",
    "severity": null,
    "message": "Environment name cannot start with the prefix AzureML. To alter a curated environment first create a copy of it.",
    "messageFormat": null,
    "messageParameters": null,
    "referenceCode": null,
    "detailsUri": null,
    "target": "EnvironmentDefinition",
    "details": [
      {
        "code": "Invalid",
        "severity": null,
        "message": "Environment name cannot start with the prefix AzureML. To alter a curated environment first create a copy of it.",
        "messageFormat": null,
        "messageParameters": {},
        "referenceCode": null,
        "detailsUri": null,
        "target": "EnvironmentDefinition",
        "details": [],
        "innerError": null,
        "debugInfo": null,
        "additionalInfo": null
      }
    ],
    "innerError": null,
    "debugInfo": null,
    "additionalInfo": null
  },
  

In [2]:
from azureml.core import Environment, Model

# ...existing code...
# ws = Workspace.get(...)
# environment_name = "env-4-housing"
# model_name = "..."

print(f"Workspace: {ws.name} | RG: {ws.resource_group} | Subscription: {ws.subscription_id}")

# 1) Environment: retrieve existing only
env_map = Environment.list(workspace=ws)  # dict-like
if environment_name not in env_map:
    available = ", ".join(sorted(env_map.keys())[:20])
    raise ValueError(
        f"Environment '{environment_name}' not found in workspace '{ws.name}'. "
        f"Available (first 20): {available}"
    )

# Get latest registered version for that name
entry = env_map[environment_name]
if isinstance(entry, list) and len(entry) > 0:
    # handle SDK variants that return a list of Environment objects
    versions = [int(e.version) for e in entry if str(e.version).isdigit()]
    version_to_use = str(max(versions)) if versions else str(entry[-1].version)
elif hasattr(entry, "version"):
    # handle SDK variants that return a single Environment object
    version_to_use = str(entry.version)
else:
    version_to_use = None

virtual_env = Environment.get(
    workspace=ws,
    name=environment_name,
    version=version_to_use
)

print(f"Using environment: {virtual_env.name}:{virtual_env.version}")

# 2) Model: retrieve existing only
registered_model = Model(ws, name=model_name)  # or Model.get(ws, name=model_name)
print(f"Using model: {registered_model.name}:{registered_model.version}")

# ...existing code...

Workspace: workspace3 | RG: Papus | Subscription: b60a6d70-289f-4f47-8d93-8fe1a73a283a
Using environment: AzureML-ACPT-pytorch-1.13-py38-cuda11.7-gpu:10
Using model: model:1


In [ ]:
try:
    service.wait_for_deployment(show_output=True)
except Exception:
    print(service.get_logs())
    raise
scoring_uri = service.scoring_uri

scoreuri = json.dumps({"URI": [scoring_uri]})
file = open("uri.json", "w")
file.write(scoreuri)
file.close()

Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2026-04-14 14:01:46-06:00 Creating Container Registry if not exists.
2026-04-14 14:01:46-06:00 Use the existing image.
2026-04-14 14:01:49-06:00 Submitting deployment to compute.
2026-04-14 14:01:54-06:00 Checking the status of deployment service..
2026-04-14 14:08:02-06:00 Checking the status of inference endpoint service.
Failed


Service deployment polling reached non-successful terminal state, current service state: Failed
Operation ID: 5a87390e-1189-422b-b392-d5b14c26ebb1
More information can be found using '.get_logs()'
Error:
{
  "code": "AciDeploymentFailed",
  "statusCode": 400,
  "message": "Aci Deployment failed with exception: Error in entry script, ModuleNotFoundError: No module named 'joblib', please run print(service.get_logs()) to get details.",
  "details": [
    {
      "code": "CrashLoopBackOff",
      "message": "Error in entry script, ModuleNotFoundError: No module named 'joblib', please run print(service.get_logs()) to get details."
    }
  ]
}



WebserviceException: WebserviceException:
	Message: Service deployment polling reached non-successful terminal state, current service state: Failed
Operation ID: 5a87390e-1189-422b-b392-d5b14c26ebb1
More information can be found using '.get_logs()'
Error:
{
  "code": "AciDeploymentFailed",
  "statusCode": 400,
  "message": "Aci Deployment failed with exception: Error in entry script, ModuleNotFoundError: No module named 'joblib', please run print(service.get_logs()) to get details.",
  "details": [
    {
      "code": "CrashLoopBackOff",
      "message": "Error in entry script, ModuleNotFoundError: No module named 'joblib', please run print(service.get_logs()) to get details."
    }
  ]
}
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Service deployment polling reached non-successful terminal state, current service state: Failed\nOperation ID: 5a87390e-1189-422b-b392-d5b14c26ebb1\nMore information can be found using '.get_logs()'\nError:\n{\n  \"code\": \"Aci